In [1]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
train_dataset = load_dataset("danish-foundation-models/danish-dynaword","ai-aktindsigt")

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

first_two_rows = train_dataset['train'].select([0])


from datasets import DatasetDict, Dataset

original_dataset = train_dataset["train"]

messages_data = {
    "messages": [[{"role": "user", "content": row["text"]}, {"role": "system", "content": ""}] for row in first_two_rows]
}

new_dataset = Dataset.from_dict(messages_data)


from datasets import DatasetDict, Dataset

rain_dataset = Dataset.from_dict(
    {
        "messages": [
            [
                {"role": "user", "content": "Vallensbæk Stationstorv 100 2665 Vallensbæk Strand Telefon: +45 4797 4000"},
            ]
        ]
    }
)

In [2]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/gemma-3-1b-pt")
model = AutoModelForCausalLM.from_pretrained("google/gemma-3-1b-pt")


def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=32768,  # Adjust this value as needed
        padding="max_length"
    )

tokenized_dataset = train_dataset.map(tokenize_function, batched=True)


In [3]:
from transformers import DataCollatorWithPadding, AutoTokenizer
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)


In [4]:
 # Tokenize
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=1024)

tokenized_dataset = train_dataset.map(tokenize, batched=True, remove_columns=train_dataset["train"].column_names)

# add already here the column of labels=input_ids, maybe through a DataCollar. Use the DataCollatorForLanguageModeling. Mybae check how to let it handle the tokenization.

check casuallanguagemodeling from hugging face and see if we can pass the labels as the logits of the teacher if this doesn't work out.

In [5]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 200914
    })
})

In [6]:
from datasets import Dataset
#from trl import GKDConfig, GKDTrainer
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
)
from trl.trainer.gkd_trainer import GKDTrainer, GKDConfig


teacher_model = AutoModelForCausalLM.from_pretrained("google/gemma-3-1b-pt")


training_args = GKDConfig(output_dir="gkd-model", per_device_train_batch_size=1, dataset_kwargs = {"skip_prepare_dataset": True})
trainer = GKDTrainer(
    model=model,
    teacher_model=teacher_model,
    #data_collator=data_collator,
    args=training_args,
    processing_class=tokenizer,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["train"],
    chatdata=False
    
)


In [ ]:
trainer.train()

It is strongly recommended to train Gemma3 models with the `eager` attention implementation instead of `sdpa`. Use `eager` with `AutoModelForCausalLM.from_pretrained('<path-to-checkpoint>', attn_implementation='eager')`.


Step,Training Loss


`generation_config` default values have been modified to match model-specific defaults: {'cache_implementation': 'hybrid', 'top_p': 0.95, 'bos_token_id': 2}. If this is not desired, please set these values explicitly.


In [ ]:
trainer.train()